# Chronos-2 Quantisation

In [ ]:
import os
from pathlib import Path

import torch

import time

os.environ.setdefault("HOME", os.environ.get("USERPROFILE", str(Path.home())))

from chop.models import get_model
from chop.ir.graph import MaseGraph
from chop.passes.graph import PASSES


In [ ]:
# Basic run config
OUTPUT_DIR = Path("artifacts")
GRAPH_NAME = "chronos2_quantized"
DEVICE = "cpu"
WRITE_QUANT_SUMMARY = False  # Set True to generate comparison CSVs

# Conservative starting quantisation config (only core arithmetic ops)
QUANT_PASS_ARGS = {
    "by": "type",
    "default": {"config": {"name": None}},
    "linear": {
        "config": {
            "name": "integer",
            "data_in_width": 8,
            "data_in_frac_width": 4,
            "weight_width": 8,
            "weight_frac_width": 4,
            "bias_width": 8,
            "bias_frac_width": 4,
        }
    },
    "matmul": {
        "config": {
            "name": "integer",
            "data_in_width": 8,
            "data_in_frac_width": 4,
            "weight_width": 8,
            "weight_frac_width": 4,
        }
    },
}

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
# 1) Load model
model = get_model("chronos-2", pretrained=True)
model.eval()
if DEVICE:
    if DEVICE.startswith("cuda") and not torch.cuda.is_available():
        raise RuntimeError("CUDA requested but torch.cuda.is_available() is False.")
    model = model.to(DEVICE)

# Force eager attention so FX sees matmul/add/softmax nodes.
if hasattr(model.config, "_attn_implementation"):
    model.config._attn_implementation = "eager"

print("Loaded Chronos-2 model")

In [ ]:
# 2) Build graph + required metadata
batch_size = 1

print(model.config.chronos_config)
c_len = model.config.chronos_config["context_length"]
out_patch = model.config.chronos_config["output_patch_size"]

dummy_in = {
    "context": torch.randn((batch_size, c_len)),
    "group_ids": torch.zeros((batch_size,), dtype=torch.long),
    "future_covariates": torch.zeros((batch_size, out_patch), dtype=torch.float32),
    "num_output_patches": 1,
}

mg = MaseGraph(
    model,
    hf_input_names=[
        "context",
        "group_ids",
        "future_covariates",
        "num_output_patches",
    ],
)

mg, _ = PASSES["init_metadata"](mg)
mg, _ = PASSES["add_common_metadata"](mg, pass_args={"dummy_in": dummy_in})

print(f"Graph nodes before quantisation: {sum(1 for _ in mg.nodes)}")

In [ ]:
# 3) Quantise
orig_mg = None
if WRITE_QUANT_SUMMARY:
    # Needed only for before/after comparison in summarize_quantization.
    orig_mg = MaseGraph(model, hf_input_names=["context", "group_ids", "future_covariates", "num_output_patches"])
    orig_mg, _ = PASSES["init_metadata"](orig_mg)
    orig_mg, _ = PASSES["add_common_metadata"](orig_mg, pass_args={"dummy_in": dummy_in})

mg, _ = PASSES["quantize"](mg, pass_args=QUANT_PASS_ARGS)
print("Quantisation pass complete")

if WRITE_QUANT_SUMMARY:
    PASSES["summarize_quantization"](
        mg,
        {"save_dir": OUTPUT_DIR / "quantize_summary", "original_graph": orig_mg},
    )
    print("Saved quantization summary to artifacts/quantize_summary")


In [ ]:
# 4) Export quantised graph
base = OUTPUT_DIR / GRAPH_NAME
mg.export(str(base))
print(f"Exported: {base}.pt and {base}.mz")

In [ ]:
# Perform bench 
import fev
from chronos import Chronos2Pipeline

# Add chronos_config to mg.model
setattr(mg.model, "config", model.config)
setattr(mg.model, "chronos_config", model.chronos_config)
setattr(mg.model, "device", model.device)

# The GraphModule returns a plain dict; wrap forward() to restore Chronos2Output type
from chop.models.chronos2.modeling_chronos2 import Chronos2Output

_orig_forward = mg.model.forward
def _patched_forward(*args, **kwargs):
    output = _orig_forward(*args, **kwargs)
    if isinstance(output, dict) and not isinstance(output, Chronos2Output):
        return Chronos2Output(**output)
    return output
mg.model.forward = _patched_forward

# Define the benchmark task
tasks_configs = [
    {
        "dataset_path": "autogluon/chronos_datasets",
        "dataset_config": "monash_m1_quarterly",
        "horizon": 8,
        "seasonality": 4,
        "eval_metric": "MASE",
    },
    {
        "dataset_path": "autogluon/chronos_datasets",
        "dataset_config": "monash_electricity_weekly",
        "horizon": 8,
        "num_windows": 2,
    },
    {
        "dataset_path": "autogluon/chronos_datasets",
        "dataset_config": "monash_m1_yearly",
        "horizon": 6,
    },
]
benchmark = fev.Benchmark.from_list(tasks_configs)

summaries = []

In [ ]:
# Run evaluation with mg.model
import tqdm
pipeline = Chronos2Pipeline(model=mg.model)

for task in tqdm.tqdm(benchmark.tasks, desc="Evaluating tasks"):
    predictions_per_window, inference_time_s = pipeline.predict_fev(task, batch_size=256)
    print(f"Inference time: {inference_time_s:.2f}s")
    summary = task.evaluation_summary(predictions_per_window, "quantised_chronos2")
    summaries.append(summary)


In [ ]:
# Predict with pre-quantisation model for comparison
pipeline = Chronos2Pipeline(model=model)
for task in tqdm.tqdm(benchmark.tasks, desc="Evaluating tasks with pre-quantisation model"):
    predictions_per_window, inference_time_s = pipeline.predict_fev(task, batch_size=256)
    print(f"Inference time (pre-quantisation model): {inference_time_s:.2f}s")
    summary = task.evaluation_summary(predictions_per_window, "pre_quantisation_chronos2")
    summaries.append(summary)

In [ ]:
fev.leaderboard(summaries=summaries, baseline_model="pre_quantisation_chronos2")

In [ ]:
# Print test errors for each task
for summary in summaries:
    print(f"Model: {summary['model_name']} Dataset: {summary['dataset_config']} test_error: {summary['test_error']:.4f}")